<a href="https://colab.research.google.com/github/jnowak123/Pigeon_VS_AI/blob/master/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pigeon VS AI — Full Pipeline

End-to-end: parse data → slide-level split → augmentation + folds → train model.

Set `MODEL_NAME` in the training config to select which model to train (M1–M5).
Run all cells top to bottom.

---
## Setup

### Imports and data path

Flip `USE_COLAB = True` for Google Colab. It will mount Drive and extract the archive if needed.

In [2]:
import io
import os
import re
import random
import tarfile

import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image, ImageFilter
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split, StratifiedKFold

# ---------- toggle ----------
USE_COLAB = True

if USE_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks")
    ARCHIVE = PROJECT_ROOT / "BreaKHis_v1.tar.gz"
    DATA_ROOT = PROJECT_ROOT / "BreaKHis_v1"
    if not DATA_ROOT.exists():
        print(f"Extracting {ARCHIVE.name}...")
        with tarfile.open(ARCHIVE) as tar:
            tar.extractall(path=PROJECT_ROOT)
        print("Done.")
    else:
        print("Already extracted.")
else:
    PROJECT_ROOT = Path(".")
    DATA_ROOT = Path("BreaKHis_v1")

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

assert DATA_ROOT.exists(), f"Data not found at {DATA_ROOT}"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting BreaKHis_v1.tar.gz...


/tmp/ipykernel_2424/658801379.py:28: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=PROJECT_ROOT)


Done.
Device: cpu


---
## Step 1 — Group Images by Slide ID

### Parse every filename into structured fields

Each BreakHis filename follows the format `SOB_{B|M}_{TYPE}-{YEAR}-{PATIENT_ID}-{MAG}-{SEQ}.png`. The regex extracts:
- **label** — benign (`B`) or malignant (`M`)
- **subtype** — tumor subtype code (e.g. `DC` = ductal carcinoma, `TA` = tubular adenoma)
- **slide_id** — `YEAR-PATIENT_ID` combination, the unique slide identifier
- **magnification** — 40, 100, 200, or 400
- **seq** — image sequence number within that slide/magnification

Any filename that doesn't match the expected pattern is collected in `unparsed` for inspection.

In [3]:
FILENAME_RE = re.compile(
    r"^SOB_([BM])_(\w+)-(\d+-\w+)-(\d+)-(\d+)\.png$"
)

records = []
unparsed = []

for png in DATA_ROOT.rglob("*.png"):
    m = FILENAME_RE.match(png.name)
    if not m:
        unparsed.append(png.name)
        continue
    class_code, subtype_code, slide_id, mag, seq = m.groups()
    label = "benign" if class_code == "B" else "malignant"
    records.append({
        "path": str(png),
        "filename": png.name,
        "label": label,
        "subtype": subtype_code,
        "slide_id": slide_id,
        "magnification": int(mag),
        "seq": int(seq),
    })

df = pd.DataFrame(records)
print(f"Parsed {len(df)} images, {len(unparsed)} unparsed")
if unparsed:
    print("Unparsed samples:", unparsed[:10])

Parsed 7909 images, 0 unparsed


### Group by slide and summarize

In [4]:
slides = df.groupby("slide_id").agg(
    label=pd.NamedAgg(column="label", aggfunc="first"),
    subtype=pd.NamedAgg(column="subtype", aggfunc="first"),
    n_images=pd.NamedAgg(column="filename", aggfunc="count"),
    magnifications=pd.NamedAgg(column="magnification", aggfunc=lambda x: sorted(x.unique().tolist())),
).reset_index()

print(f"{len(slides)} unique slides")
print(f"\nSlides per class:")
print(slides["label"].value_counts())
print(f"\nSlides per subtype:")
print(slides["subtype"].value_counts())

mag_counts = df.groupby(["label", "magnification"]).size().unstack(fill_value=0)
print(f"\nImages per class × magnification:")
print(mag_counts)

missing_mags = slides[slides["magnifications"].apply(lambda m: set(m) != {40, 100, 200, 400})]
if len(missing_mags) == 0:
    print(f"\nAll slides have all 4 magnification levels.")
else:
    print(f"\n{len(missing_mags)} slides missing magnification levels:")
    print(missing_mags)

print(f"\nImages per slide — min: {slides['n_images'].min()}, "
      f"max: {slides['n_images'].max()}, "
      f"median: {slides['n_images'].median():.0f}")

81 unique slides

Slides per class:
label
malignant    57
benign       24
Name: count, dtype: int64

Slides per subtype:
subtype
DC    38
F     10
MC     9
TA     7
PC     6
LC     4
A      4
PT     3
Name: count, dtype: int64

Images per class × magnification:
magnification   40    100   200   400
label                                
benign          625   644   623   588
malignant      1370  1437  1390  1232

All slides have all 4 magnification levels.

Images per slide — min: 38, max: 246, median: 90


---
## Step 2 — Slide-Level Stratified Split

### Split slides 1:7 (test : train-val), stratified by label

The test set (~1,000 images) is held out entirely and never used during training or validation.

In [5]:
SEED = 42
TEST_RATIO = 1 / 8

train_val_slides, test_slides = train_test_split(
    slides,
    test_size=TEST_RATIO,
    stratify=slides["label"],
    random_state=SEED,
)

test_slide_ids = set(test_slides["slide_id"])
df["split"] = df["slide_id"].apply(
    lambda s: "test" if s in test_slide_ids else "train_val"
)

print(f"Train-val: {len(train_val_slides)} slides, Test: {len(test_slides)} slides")
print(f"\nImages per split:")
print(df["split"].value_counts())
print(f"\nImages per split × class:")
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))

Train-val: 70 slides, Test: 11 slides

Images per split:
split
train_val    6801
test         1108
Name: count, dtype: int64

Images per split × class:
label      benign  malignant
split                       
test          282        826
train_val    2198       4603


### Verify no slide leakage

In [6]:
train_val_ids = set(train_val_slides["slide_id"])
overlap = test_slide_ids & train_val_ids
assert len(overlap) == 0, f"LEAKAGE: {overlap}"
print(f"No leakage — 0 slides shared between test and train-val.")

print(f"\nTest slides:")
print(test_slides[["slide_id", "label", "subtype", "n_images"]].sort_values("label").to_string(index=False))

No leakage — 0 slides shared between test and train-val.

Test slides:
  slide_id     label subtype  n_images
  14-22704    benign      PT       158
14-29315EF    benign      PT        60
 14-19854C    benign      TA        64
  14-16196 malignant      LC        74
  14-15704 malignant      PC       127
   14-2773 malignant      DC       144
  14-20636 malignant      DC       111
14-13418DE malignant      MC        55
  14-16456 malignant      MC       178
  14-16448 malignant      DC        53
   14-4372 malignant      DC        84


---
## Step 3 — Augmentation Manifest + 5-Fold CV

### Expand train-val images with augmentation tags

Each image gets 6 rows: original + rot90/180/270 + flip_h/flip_v.
No pixels are copied — the dataloader applies transforms on-the-fly using the `augmentation` column.

In [7]:
AUGMENTATIONS = ["original", "rot90", "rot180", "rot270", "flip_h", "flip_v"]

train_val_images = df[df["split"] == "train_val"].copy()
test_images = df[df["split"] == "test"].copy()

rows = []
for _, row in train_val_images.iterrows():
    for aug in AUGMENTATIONS:
        new_row = row.to_dict()
        new_row["augmentation"] = aug
        rows.append(new_row)

train_aug = pd.DataFrame(rows)
print(f"Augmented train-val: {len(train_aug)} rows ({len(train_val_images)} × {len(AUGMENTATIONS)})")

Augmented train-val: 40806 rows (6801 × 6)


### Assign 5-fold CV at the slide level

Each slide maps to exactly one fold via `StratifiedKFold`. All images and augmentations from a slide share the same fold — no leakage within folds.

In [8]:
N_FOLDS = 5

slides_tv = train_val_images[["slide_id", "label"]].drop_duplicates().reset_index(drop=True)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
slide_to_fold = {}
for fold, (_, val_idx) in enumerate(skf.split(slides_tv, slides_tv["label"])):
    for sid in slides_tv.loc[val_idx, "slide_id"]:
        slide_to_fold[sid] = fold

train_aug["fold"] = train_aug["slide_id"].map(slide_to_fold)

# verify
for f in range(N_FOLDS):
    val_slides = set(train_aug[train_aug["fold"] == f]["slide_id"])
    train_slides = set(train_aug[train_aug["fold"] != f]["slide_id"])
    assert len(val_slides & train_slides) == 0, f"LEAKAGE in fold {f}"
print(f"No leakage — all {N_FOLDS} folds clean.")

print(f"\nAugmented images per fold:")
for f in range(N_FOLDS):
    val_count = len(train_aug[train_aug["fold"] == f])
    train_count = len(train_aug[train_aug["fold"] != f])
    print(f"  Fold {f}: train={train_count}, val={val_count}")

No leakage — all 5 folds clean.

Augmented images per fold:
  Fold 0: train=33270, val=7536
  Fold 1: train=31674, val=9132
  Fold 2: train=31704, val=9102
  Fold 3: train=32826, val=7980
  Fold 4: train=33750, val=7056


### Save manifests to disk

Persist the augmented train manifest and test manifest as CSVs.
These serve as checkpoints — if the notebook restarts, training can resume from here.

In [9]:
test_images["augmentation"] = "original"
test_images["fold"] = -1

train_aug.to_csv(DATA_DIR / "train_manifest.csv", index=False)
test_images.to_csv(DATA_DIR / "test_manifest.csv", index=False)
df.to_csv(DATA_DIR / "image_catalog.csv", index=False)
slides.to_csv(DATA_DIR / "slide_summary.csv", index=False)

print(f"Saved train_manifest.csv ({len(train_aug)} rows)")
print(f"Saved test_manifest.csv ({len(test_images)} rows)")
print(f"Saved image_catalog.csv, slide_summary.csv")

Saved train_manifest.csv (40806 rows)
Saved test_manifest.csv (1108 rows)
Saved image_catalog.csv, slide_summary.csv


---
## Step 4 — Model Training (5-Fold CV)

### Training configuration

Set `MODEL_NAME` to one of `M1`–`M5`.

| Model | Magnification | Architecture     | Notes |
|-------|---------------|------------------|-------|
| M1    | 40X           | ResNet50         | Most spatial context |
| M2    | 100X          | EfficientNet-B3  | Balanced zoom |
| M3    | 200X          | EfficientNet-B4  | Higher cell detail |
| M4    | 400X          | Swin Transformer | Finest detail, global attention |
| M5    | All (40–400X) | ResNet50         | Generalist + blur/compression augmentation |

In [10]:
MODEL_NAME = "M1"

MODEL_CONFIGS = {
    "M1": {"arch": "resnet50",         "magnification": 40,   "input_size": 224, "blur_compress": False},
    "M2": {"arch": "efficientnet_b3",  "magnification": 100,  "input_size": 224, "blur_compress": False},
    "M3": {"arch": "efficientnet_b4",  "magnification": 200,  "input_size": 224, "blur_compress": False},
    "M4": {"arch": "swin_t",           "magnification": 400,  "input_size": 224, "blur_compress": False},
    "M5": {"arch": "resnet50",         "magnification": None, "input_size": 224, "blur_compress": True},
}

BATCH_SIZE = 32
NUM_EPOCHS = 30
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 5
NUM_WORKERS = 2

cfg = MODEL_CONFIGS[MODEL_NAME]
print(f"Training {MODEL_NAME}: arch={cfg['arch']}, mag={cfg['magnification']}, blur_compress={cfg['blur_compress']}")

Training M1: arch=resnet50, mag=40, blur_compress=False


### Dataset class

Reads images, applies geometric augmentations from the `augmentation` column on-the-fly,
and optionally adds blur + JPEG compression for M5. Resizes to 224×224 and normalizes to ImageNet stats.

In [11]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

GEOMETRIC_OPS = {
    "original": lambda img: img,
    "rot90": lambda img: img.rotate(-90, expand=True),
    "rot180": lambda img: img.rotate(-180),
    "rot270": lambda img: img.rotate(-270, expand=True),
    "flip_h": lambda img: img.transpose(Image.FLIP_LEFT_RIGHT),
    "flip_v": lambda img: img.transpose(Image.FLIP_TOP_BOTTOM),
}


class BreakHisDataset(Dataset):
    def __init__(self, manifest, magnification=None, input_size=224, blur_and_compress=False):
        df = manifest.copy()
        if magnification is not None:
            df = df[df["magnification"] == magnification]
        self.records = df.reset_index(drop=True)
        self.blur_and_compress = blur_and_compress
        self.resize_and_normalize = transforms.Compose([
            transforms.Resize((input_size, input_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        aug_fn = GEOMETRIC_OPS.get(row["augmentation"], GEOMETRIC_OPS["original"])
        img = aug_fn(img)
        if self.blur_and_compress and row["augmentation"] != "original":
            img = self._apply_blur_compress(img)
        img = self.resize_and_normalize(img)
        label = 1 if row["label"] == "malignant" else 0
        return img, label

    @staticmethod
    def _apply_blur_compress(img):
        if random.random() < 0.3:
            radius = random.uniform(0.5, 1.5)
            img = img.filter(ImageFilter.GaussianBlur(radius=radius))
        if random.random() < 0.3:
            quality = random.randint(30, 70)
            buf = io.BytesIO()
            img.save(buf, format="JPEG", quality=quality)
            buf.seek(0)
            img = Image.open(buf).convert("RGB")
        return img

### Model builder

Construct the selected architecture with pretrained ImageNet weights.
The final classification head is replaced with a single output neuron for binary classification.

In [12]:
def build_model(arch):
    if arch == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, 1)
    elif arch == "efficientnet_b3":
        model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 1)
    elif arch == "efficientnet_b4":
        model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 1)
    elif arch == "swin_t":
        model = models.swin_t(weights=models.Swin_T_Weights.DEFAULT)
        model.head = nn.Linear(model.head.in_features, 1)
    else:
        raise ValueError(f"Unknown architecture: {arch}")
    return model

test_model = build_model(cfg["arch"])
total_params = sum(p.numel() for p in test_model.parameters())
print(f"Architecture: {cfg['arch']}, params: {total_params:,}")
del test_model

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 150MB/s]


Architecture: resnet50, params: 23,510,081


### Training and validation functions

In [13]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.float().to(device)
        optimizer.zero_grad()
        logits = model(imgs).squeeze(1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        preds = (logits > 0).long()
        correct += (preds == labels.long()).sum().item()
        total += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.float().to(device)
        logits = model(imgs).squeeze(1)
        loss = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        preds = (logits > 0).long()
        correct += (preds == labels.long()).sum().item()
        total += len(labels)
    return total_loss / total, correct / total

### 5-fold training loop

For each fold:
1. Build a fresh model with pretrained weights
2. Train on 4 folds, validate on 1
3. Early stopping on validation loss (patience=5)
4. Save best checkpoint to `models/{MODEL_NAME}_fold{f}.pt`

In [14]:
manifest = train_aug
fold_results = []

for fold in range(N_FOLDS):
    print(f"\n{'='*60}")
    print(f"Fold {fold}/{N_FOLDS - 1}")
    print(f"{'='*60}")

    train_df = manifest[manifest["fold"] != fold]
    val_df = manifest[manifest["fold"] == fold]

    train_ds = BreakHisDataset(
        train_df,
        magnification=cfg["magnification"],
        input_size=cfg["input_size"],
        blur_and_compress=cfg["blur_compress"],
    )
    val_ds = BreakHisDataset(
        val_df,
        magnification=cfg["magnification"],
        input_size=cfg["input_size"],
        blur_and_compress=False,
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    print(f"Train: {len(train_ds)} images, Val: {len(val_ds)} images")

    model = build_model(cfg["arch"]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    criterion = nn.BCEWithLogitsLoss()

    best_val_loss = float("inf")
    patience_counter = 0
    save_path = MODEL_DIR / f"{MODEL_NAME}_fold{fold}.pt"

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = validate(model, val_loader, criterion)
        scheduler.step()

        print(f"  Epoch {epoch:2d} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))
    final_val_loss, final_val_acc = validate(model, val_loader, criterion)
    fold_results.append({"fold": fold, "val_loss": final_val_loss, "val_acc": final_val_acc})
    print(f"  Best — val_loss={final_val_loss:.4f} val_acc={final_val_acc:.4f} → saved {save_path.name}")

    del model, optimizer, scheduler
    torch.cuda.empty_cache()


Fold 0/4
Train: 8382 images, Val: 1986 images


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

### Cross-validation summary

In [ ]:
results_df = pd.DataFrame(fold_results)
print(f"\n{MODEL_NAME} — 5-Fold CV Results")
print(f"{'='*40}")
print(results_df.to_string(index=False))
print(f"\nMean val_acc: {results_df['val_acc'].mean():.4f} ± {results_df['val_acc'].std():.4f}")
print(f"Mean val_loss: {results_df['val_loss'].mean():.4f} ± {results_df['val_loss'].std():.4f}")

results_df.to_csv(DATA_DIR / f"{MODEL_NAME}_cv_results.csv", index=False)
print(f"\nSaved to {DATA_DIR / f'{MODEL_NAME}_cv_results.csv'}")